In [1]:
!source /venv/main/bin/activate
!/venv/main/bin/python -m pip install kagglehub pandas transformers sklearn accelerate
!export KAGGLE_API_TOKEN=KGAT_57afc4e7bca72ee06ff06959f2bbdeb6
# # IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# # RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
# import kagglehub
# kagglehub.login()


Activated conda/uv virtual environment at /venv/main


ERROR: Could not find a version that satisfies the requirement skplearn (from versions: none)
ERROR: No matching distribution found for skplearn


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# jakeclark38a_smart_contract_vulnerability_detection_path = kagglehub.dataset_download('jakeclark38a/smart-contract-vulnerability-detection', output_dir="./dataset")

# print(f'Data source import complete. {jakeclark38a_smart_contract_vulnerability_detection_path}')


# Smart Contract Vulnerability Detection - GPT-2 Fine-tuning
Training with all optimization thresholds: before_optimized, optimized_80p, optimized_50p, optimized_20p

Uses GPT-2 with prompting to generate structured Markdown vulnerability reports.
Example output format:
Vulnerabilities:
* Arithmetic
* Denial of Service

In [3]:
import os, sys, time, subprocess
import torch

print('='*60)
print('ENVIRONMENT CHECK')
print('='*60)
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print()

ENVIRONMENT CHECK
Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4090



In [4]:
print('='*60)
print('STEP 1: Install Dependencies')
print('='*60)

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'uv', '-q'],
    capture_output=True, text=True, timeout=60
)
print(f'uv install: OK' if result.returncode == 0 else 'FAILED')

missing_packages = ['transformers', 'accelerate', 'datasets', 'scikit-learn']
result = subprocess.run(
    ['uv', 'pip', 'install', '--system'] + missing_packages,
    capture_output=True, text=True, timeout=600
)
print(f'Packages install: OK' if result.returncode == 0 else 'FAILED')

for pkg in ['transformers', 'pandas', 'sklearn']:
    try:
        mod = __import__(pkg)
        print(f'  [OK] {pkg}')
    except ImportError:
        print(f'  [MISSING] {pkg}')
print()

STEP 1: Install Dependencies


uv install: OK
Packages install: OK


  [OK] transformers


  [OK] pandas


  [OK] sklearn



In [5]:
print('='*60)
print('STEP 2: Load Optimized Dataset (FULL)')
print('='*60)

import pandas as pd
from torch.utils.data import DataLoader
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report, hamming_loss

dataset_dir = '/workspace/dataset'

train_df = pd.read_csv(os.path.join(dataset_dir, 'train_optimized_dataset.csv'))
test_df = pd.read_csv(os.path.join(dataset_dir, 'test_optimized_dataset.csv'))

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Columns: {train_df.columns.tolist()}')

label_columns = ['Arithmetic', 'Unchecked Return Values For Low Level Calls',
                 'Denial of Service', 'Time manipulation', 'Reentrancy']
print(f'Labels: {label_columns}')
print()

STEP 2: Load Optimized Dataset (FULL)


Train samples: 8444
Test samples: 2111
Columns: ['address', 'before_optimized', 'optimized_80p', 'optimized_50p', 'optimized_20p', 'Arithmetic', 'Unchecked Return Values For Low Level Calls', 'Denial of Service', 'Time manipulation', 'Reentrancy']
Labels: ['Arithmetic', 'Unchecked Return Values For Low Level Calls', 'Denial of Service', 'Time manipulation', 'Reentrancy']



In [6]:
print('='*60)
print('STEP 3: Define Helper Functions')
print('='*60)

import gc
import numpy as np
from sklearn.metrics import classification_report, hamming_loss, precision_recall_fscore_support
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset

def format_vulnerabilities(labels, label_columns):
    vulns = []
    for i, label in enumerate(label_columns):
        if labels[i] == 1:
            vulns.append(label)
    if not vulns:
        vulns.append('None')
    return 'Vulnerabilities:\n' + '\n'.join(f'* {v}' for v in vulns)

def create_prompt(text):
    # Simply return the code. No extra text needed.
    return text[:1024] # GPT-2 limit

def parse_vulnerabilities(output_text, label_columns):
    output_lower = output_text.lower()
    detected = []
    for label in label_columns:
        label_lower = label.lower()
        if label_lower in output_lower:
            detected.append(1)
        else:
            detected.append(0)
    if sum(detected) == 0:
        detected = [0] * len(label_columns)
    return detected

class VulnerabilityClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=1024):
        self.texts = texts
        # Labels must be float32 for multi-label BCE loss
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.texts)

def hamming_score(y_true, y_pred, normalize=True, sample_weight=None):
    acc_list = []
    for i in range(y_true.shape[0]):
        set_true = set(np.where(y_true[i])[0])
        set_pred = set(np.where(y_pred[i])[0])
        tmp_a = None
        if len(set_true) == 0 and len(set_pred) == 0:
            tmp_a = 1
        else:
            tmp_a = len(set_true.intersection(set_pred))/float(len(set_true.union(set_pred)))
        acc_list.append(tmp_a)
    return np.mean(acc_list)

def evaluate_model(model, test_dataset, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_labels = []

    print(f"Starting inference on {len(test_dataset)} samples...")

    with torch.no_grad():
        for batch in test_loader:
            # Move batch to GPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Convert logits to probabilities using Sigmoid
            # Formula: 1 / (1 + exp(-x))
            probs = torch.sigmoid(logits)

            # Threshold at 0.5 to get binary predictions
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Flatten results
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    return y_true, y_pred

def train_and_evaluate(column_name, train_df, test_df, label_columns, output_dir):
    print(f'\n' + '='*60)
    print(f'Training with column: {column_name}')
    print('='*60)

    train_texts = train_df[column_name].fillna('').astype(str).tolist()
    test_texts = test_df[column_name].fillna('').astype(str).tolist()
    train_labels = train_df[label_columns].values
    test_labels = test_df[label_columns].values

    print(f'Train: {len(train_texts)}, Test: {len(test_texts)}')

    MODEL_NAME = 'gpt2'
    tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left" # CRITICAL for GPT-2 classification

    train_dataset = VulnerabilityClassificationDataset(train_texts, train_labels, tokenizer, max_length=1024)
    test_dataset = VulnerabilityClassificationDataset(test_texts, test_labels, tokenizer, max_length=1024)

    model = GPT2ForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_columns),
        problem_type="multi_label_classification" # This automatically sets the right loss
    )
    model.config.pad_token_id = tokenizer.eos_token_id

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    train_size = int(0.8 * len(train_dataset))
    eval_size = len(train_dataset) - train_size
    train_dataset_split, eval_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, eval_size]
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=10,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=False,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to='none',
        gradient_accumulation_steps=2,
        eval_accumulation_steps=10,  # Move to CPU every 10 batches
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_split,
        eval_dataset=eval_dataset,
    )

    train_start = time.time()
    print('Training...')
    trainer.train()
    train_time = time.time() - train_start

    print('Generating predictions...')
    test_start = time.time()
    test_labels, pred_labels = evaluate_model(model, test_dataset)
    test_inference_time = time.time() - test_start

    print('\nClassification Report by Label:')
    print(classification_report(test_labels, pred_labels, target_names=label_columns, zero_division=0))

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, pred_labels, average='weighted', zero_division=0
    )
    hamming = hamming_score(test_labels, pred_labels)
    h_loss = hamming_loss(test_labels, pred_labels)

    model_save_dir = os.path.join(output_dir, column_name)
    os.makedirs(model_save_dir, exist_ok=True)
    model.save_pretrained(model_save_dir)
    tokenizer.save_pretrained(model_save_dir)

    del model, trainer, train_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'column': column_name,
        'train_time': train_time,
        'train_inference_time': 0,
        'test_inference_time': test_inference_time,
        'num_train_samples': len(train_texts),
        'num_test_samples': len(test_texts),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'hamming_score': hamming,
        'hamming_loss': h_loss,
    }

print('Helper functions defined')
print()

STEP 3: Define Helper Functions
Helper functions defined



In [7]:
print('='*60)
print('STEP 4: Run All Experiments')
print('='*60)

import json
from pathlib import Path

output_base = '/workspace/output'
Path(output_base).mkdir(parents=True, exist_ok=True)

columns = ['before_optimized', 'optimized_80p', 'optimized_50p', 'optimized_20p']

all_results = []
total_start = time.time()

for col in columns:
    result = train_and_evaluate(
        column_name=col,
        train_df=train_df,
        test_df=test_df,
        label_columns=label_columns,
        output_dir=output_base
    )
    all_results.append(result)

    print(f'\nResults for {col}:')
    print(f'  Train Samples: {result["num_train_samples"]}, Test Samples: {result["num_test_samples"]}')
    print(f'  Train Time: {result["train_time"]/60:.1f} min')
    print(f'  Train Inference Time: {result["train_inference_time"]:.2f}s')
    print(f'  Test Inference Time: {result["test_inference_time"]:.2f}s')
    print(f'  Precision: {result["precision"]:.4f}')
    print(f'  Recall: {result["recall"]:.4f}')
    print(f'  F1: {result["f1"]:.4f}')
    print(f'  Hamming Score: {result["hamming_score"]:.4f}')
    print(f'  Hamming Loss: {result["hamming_loss"]:.4f}')

    gc.collect()
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f'\nTotal training time: {total_time/60:.1f} minutes')
print()

STEP 4: Run All Experiments

Training with column: before_optimized
Train: 8444, Test: 2111


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training...


Epoch,Training Loss,Validation Loss
1,0.696421,0.335121
2,0.646020,0.305431
3,0.588352,0.289899
4,0.555712,0.272489
5,0.529050,0.275614
6,0.477198,0.262662
7,0.472562,0.260902
8,0.428397,0.251857
9,0.427706,0.250064
10,0.394769,0.251826


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Generating predictions...
Starting inference on 2111 samples...



Classification Report by Label:
                                             precision    recall  f1-score   support

                                 Arithmetic       0.95      0.99      0.97      1952
Unchecked Return Values For Low Level Calls       0.84      0.87      0.86      1180
                          Denial of Service       0.87      0.81      0.84       985
                          Time manipulation       0.65      0.66      0.65       668
                                 Reentrancy       0.78      0.83      0.80       826

                                  micro avg       0.85      0.87      0.86      5611
                                  macro avg       0.82      0.83      0.82      5611
                               weighted avg       0.85      0.87      0.86      5611
                                samples avg       0.85      0.86      0.84      5611



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Results for optimized_20p:
  Train Samples: 8444, Test Samples: 2111
  Train Time: 22.7 min
  Train Inference Time: 0.00s
  Test Inference Time: 13.37s
  Precision: 0.8509
  Recall: 0.8706
  F1: 0.8603
  Hamming Score: 0.7966
  Hamming Loss: 0.1500

Total training time: 96.1 minutes



In [8]:
print('='*60)
print('STEP 5: Summary Comparison')
print('='*60)

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[['column', 'num_train_samples', 'num_test_samples', 'train_time', 'train_inference_time', 'test_inference_time', 'precision', 'recall', 'f1', 'hamming_score', 'hamming_loss']]
comparison_df.columns = ['Dataset', 'Train Samples', 'Test Samples', 'Train Time (s)', 'Train Inference (s)', 'Test Inference (s)', 'Precision', 'Recall', 'F1', 'Hamming Score', 'Hamming Loss']

print('\n' + '='*100)
print('FINAL RESULTS COMPARISON')
print('='*100)
print(comparison_df.to_string(index=False))
print('='*100)

comparison_csv = os.path.join(output_base, 'comparison_results.csv')
comparison_df.to_csv(comparison_csv, index=False)
print(f'\nResults saved to: {comparison_csv}')

results_json = {
    'configuration': {
        'model': 'gpt2',
        'labels': label_columns
    },
    'results': all_results,
    'total_time': total_time
}

results_json_path = os.path.join(output_base, 'experiment_results.json')
with open(results_json_path, 'w') as f:
    json.dump(results_json, f, indent=2)
print(f'Results saved to: {results_json_path}')

print('\nAll experiments completed!')
print()

STEP 5: Summary Comparison

FINAL RESULTS COMPARISON
         Dataset  Train Samples  Test Samples  Train Time (s)  Train Inference (s)  Test Inference (s)  Precision   Recall       F1  Hamming Score  Hamming Loss
before_optimized           8444          2111     1475.756669                    0           16.058710   0.883374 0.925147 0.903603       0.847371      0.105353
   optimized_80p           8444          2111     1446.988757                    0           14.798589   0.885197 0.916236 0.900216       0.842973      0.108195
   optimized_50p           8444          2111     1406.902611                    0           14.055716   0.889978 0.889681 0.889461       0.832307      0.116532
   optimized_20p           8444          2111     1364.832972                    0           13.368172   0.850872 0.870611 0.860252       0.796589      0.149976

Results saved to: /workspace/output/comparison_results.csv
Results saved to: /workspace/output/experiment_results.json

All experiments compl